In [ ]:
# Parameters
file_path_from_event=""
context = "default_value"


In [ ]:
result_data = {
    "filename": file_path_from_event,
    "id": context,
    "status": "Success",
    "error": None    
}

continue_processing = True
print(f"file_path_from_event:===== {file_path_from_event}")

print("Читаєм context default value")
if context == "default_value" or not context:
    print("Помилка: Не отримано ідентифікатор повідомлення")
    result_data["status"] = "Error"
    result_data["error"] = "Не отримано ідентифікатор повідомлення"    
    continue_processing = False
    
else:
    print(f"Отримано ідентифікатор: {context}")






In [ ]:
from notebookutils import mssparkutils
import os
if continue_processing:
    try:
        raw_path = file_path_from_event.strip()

        # Видаляємо початковий слеш, якщо він заважає (як ви помітили з "Files")
        if raw_path.startswith("/"):
            source_path = raw_path[1:]
        else:
            source_path = raw_path

        # Шлях до папки для снепшотів
        snapshots_dir = "Files/processing_snapshots"
        print("Перевіряємо, чи існує папка, і створюємо її, якщо ні")
        if not mssparkutils.fs.exists(snapshots_dir):
            print(f"Створюю папку: {snapshots_dir}")
            mssparkutils.fs.mkdirs(snapshots_dir)
        else:
            print(f"каталог {snapshots_dir}  присутній")

        print("Копіюю файл для прийому")
        if source_path:
            print( "Створюємо ім'я для тимчасової копії, щоб уникнути конфліктів")
            file_name = os.path.basename(source_path)
            snapshot_path = f"{snapshots_dir}/snap_{file_name}"
            
            print(f"Копіюю з !{source_path}! в !{snapshot_path}!")
            # Створюємо Snapshot
            mssparkutils.fs.cp( source_path, snapshot_path)
            
            # Тепер працюємо з snapshot_path — він гарантовано не зміниться!
            print(f"Починаємо обробку ізольованої копії: {snapshot_path}")
    except Exception as e:
        result_data["status"] = "Error"
        result_data["error"] = f"FS Error: {str(e)}"
        continue_processing = False 
  
       

In [ ]:
import json
from notebookutils import mssparkutils
from pyspark.sql.functions import current_timestamp
if continue_processing:
  try:
    print(f"Відмічаю, що файл оброблено: {snapshot_path}")

    # Використовуємо синтаксис :назва_змінної всередині запиту
    spark.sql("""
        UPDATE ev2_file_grn
        SET 
          processing_status = 'PROCESSED',
          processed_at = current_timestamp()
        WHERE id = :context_id AND processing_status = 'NEW'
    """, args={"context_id": context})

    print(f"Відмічаю, що файл оброблено: {snapshot_path} - ok")
    result_data["status"] = "Success"
    result_data["snapshot"] = snapshot_path
  except Exception as e:
      result_data["status"] = "Error"
      result_data["error"] = str(e)
  
mssparkutils.notebook.exit(json.dumps(result_data))  